# Normalized Vs Unnormalized Noise Symbol Error Comparison

This notebook reproduces the experiment-style SOI/interferer symbol-error surf plots for 1, 2, and 4 channels, while comparing:

- **unnormalized noise**: the repo's default `NoiseConfig(sigma2=...)` behavior, and
- **normalized noise**: a separate analysis-only variant where the actual added noise power is scaled relative to the clean waveform power.

This notebook does **not** modify the repo's generator or training code. It only reuses the existing models and generator in a separate analysis path.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
PROJECT_ROOT = next((path for path in candidate_roots if (path / 'config.py').exists() and (path / 'pytorch_models').exists()), cwd)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt

from config import ExperimentConfig
from networks.iq_cnn_separator import IQCNNSeparator
from utils.data_utils.generator import RFMixtureGenerator, QPSKConfig, NoiseConfig, MixtureConfig
from utils.model_utils.symbol_utils import rrc_taps, recover_symbols_from_waveform
from utils.model_utils.conversion_helpers import complex_to_2ch, complex_matrix_to_iq_channels

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Helpers

The repo's built-in noise uses an **absolute complex sample variance** `sigma2`.

The normalized-noise analysis here instead uses:

`noise_power = normalized_sigma2 * signal_power`

where `signal_power = mean(|clean_mixture|^2)`. This keeps noise tied to the clean waveform normalization without changing repo behavior globally.

In [ ]:
def resolve_checkpoint(channel_count: int) -> Path:
    candidates = {
        1: [
            PROJECT_ROOT / 'pytorch_models' / 'IQ_CNN_1_channel.pt',
            PROJECT_ROOT / 'pytorch_models' / 'IQ_CNN_1_channel_backup.pt',
        ],
        2: [
            PROJECT_ROOT / 'pytorch_models' / 'IQ_CNN_2_channel.pt',
            PROJECT_ROOT / 'pytorch_models' / 'IQ_CNN_2_channel_new.pt',
            PROJECT_ROOT / 'pytorch_models' / 'IQ_CNN_2_channel_backup.pt',
        ],
        4: [
            PROJECT_ROOT / 'pytorch_models' / 'IQ_CNN_4_channel.pt',
            PROJECT_ROOT / 'pytorch_models' / 'IQ_CNN_4_channel_new.pt',
            PROJECT_ROOT / 'pytorch_models' / 'IQ_CNN_4_channel_backup.pt',
        ],
    }
    for path in candidates[channel_count]:
        if path.exists():
            return path
    raise FileNotFoundError(f'No checkpoint found for {channel_count}-channel case')


def load_separator(channel_count: int):
    ckpt = resolve_checkpoint(channel_count)
    model = IQCNNSeparator(in_ch=2 * channel_count, out_ch=4).to(device)
    model.load_state_dict(torch.load(str(ckpt), map_location=device))
    model.eval()
    return model, ckpt


def count_matches(pred_syms: np.ndarray, true_syms: np.ndarray) -> int:
    n = min(len(pred_syms), len(true_syms))
    return int(np.sum(pred_syms[:n] == true_syms[:n]))


def generate_clean_example(alpha: float, seed: int, n_rx: int, n_symbols: int, sps: int, rolloff: float, span: int, phase_shift_deg: int):
    gen = RFMixtureGenerator(seed=seed)
    qpsk_cfg = QPSKConfig(
        n_symbols=n_symbols,
        samples_per_symbol=sps,
        rolloff=rolloff,
        rrc_span_symbols=span,
        normalize_power=True,
        num_channels=n_rx,
    )
    noise_cfg = NoiseConfig(enabled=False, snr_db=None, sigma2=None)
    mix_cfg = MixtureConfig(
        alpha=float(alpha),
        snr_db=None,
        n_rx=n_rx,
        random_phase=False,
        phase_shift_deg=phase_shift_deg,
        interference_phase_shift=0,
    )
    return gen.generate_mixture(qpsk_cfg, qpsk_cfg, noise_cfg, mix_cfg)


def add_unnormalized_noise(clean_mixture: np.ndarray, sigma2: float, seed: int):
    rng = np.random.default_rng(seed)
    if sigma2 == 0.0:
        return clean_mixture.copy()
    noise = (
        rng.standard_normal(clean_mixture.shape) + 1j * rng.standard_normal(clean_mixture.shape)
    ) / np.sqrt(2.0)
    noise *= np.sqrt(sigma2)
    return clean_mixture + noise.astype(np.complex64)


def add_normalized_noise(clean_mixture: np.ndarray, normalized_sigma2: float, seed: int):
    rng = np.random.default_rng(seed)
    signal_power = float(np.mean(np.abs(clean_mixture) ** 2))
    noise_power = float(normalized_sigma2 * signal_power)
    if noise_power == 0.0:
        return clean_mixture.copy()
    noise = (
        rng.standard_normal(clean_mixture.shape) + 1j * rng.standard_normal(clean_mixture.shape)
    ) / np.sqrt(2.0)
    noise *= np.sqrt(noise_power)
    return clean_mixture + noise.astype(np.complex64)


def predict_waveforms(mixture: np.ndarray, model, n_rx: int):
    if n_rx == 1:
        x = complex_to_2ch(mixture)
    else:
        x = complex_matrix_to_iq_channels(mixture)
    x_tensor = torch.from_numpy(x).float().unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x_tensor).squeeze(0).cpu().numpy()
    pred_a = pred[0] + 1j * pred[1]
    pred_b = pred[2] + 1j * pred[3]
    return pred_a, pred_b


def count_symbol_errors(ex: dict, mixture: np.ndarray, model, taps: np.ndarray, sps: int, n_rx: int):
    pred_a, pred_b = predict_waveforms(mixture, model, n_rx)
    rec_a = recover_symbols_from_waveform(pred_a, taps, sps, len(ex['symbols_a']))
    rec_b = recover_symbols_from_waveform(pred_b, taps, sps, len(ex['symbols_b']))
    true_a = ex['symbols_a']
    true_b = ex['symbols_b']

    direct = count_matches(rec_a, true_a) + count_matches(rec_b, true_b)
    swapped = count_matches(rec_a, true_b) + count_matches(rec_b, true_a)

    if direct >= swapped:
        soi_correct = count_matches(rec_a, true_a)
        int_correct = count_matches(rec_b, true_b)
    else:
        soi_correct = count_matches(rec_b, true_a)
        int_correct = count_matches(rec_a, true_b)

    n_symbols = len(true_a)
    return n_symbols - soi_correct, n_symbols - int_correct


def run_symbol_error_sweep(model, n_rx: int, phase_shift_deg: int, alpha_values, sigma2_values, trials, n_symbols, sps, rolloff, span, noise_mode: str):
    taps = rrc_taps(sps=sps, beta=rolloff, span_symbols=span)
    soi_grid = np.zeros((len(alpha_values), len(sigma2_values)), dtype=float)
    int_grid = np.zeros((len(alpha_values), len(sigma2_values)), dtype=float)

    for ai, alpha in enumerate(alpha_values):
        for si, sigma2 in enumerate(sigma2_values):
            soi_errs = []
            int_errs = []
            for trial in range(trials):
                seed = 10000 * ai + 100 * si + trial
                ex = generate_clean_example(alpha, seed, n_rx, n_symbols, sps, rolloff, span, phase_shift_deg)
                clean = ex['mixture']
                noise_seed = seed + 500000
                if noise_mode == 'unnormalized':
                    mixture = add_unnormalized_noise(clean, sigma2, noise_seed)
                elif noise_mode == 'normalized':
                    mixture = add_normalized_noise(clean, sigma2, noise_seed)
                else:
                    raise ValueError(f'unknown noise_mode={noise_mode}')

                soi_err, int_err = count_symbol_errors(ex, mixture, model, taps, sps, n_rx)
                soi_errs.append(soi_err)
                int_errs.append(int_err)

            soi_grid[ai, si] = float(np.mean(soi_errs))
            int_grid[ai, si] = float(np.mean(int_errs))

    return soi_grid, int_grid

## Sweep Parameters

These defaults mirror the experiment-style notebooks closely enough to compare shape differences while keeping the notebook easy to rerun.

- `unnormalized_sigma2_values` matches the familiar raw-variance style from the notebooks.
- `normalized_sigma2_values` is a ratio relative to clean waveform power.

In [ ]:
alpha_values = np.round(np.arange(0.0, 2.0 + 0.1, 0.1), 1)
unnormalized_sigma2_values = np.arange(0.0, 21.0, 1.0, dtype=float)
normalized_sigma2_values = np.array([0.0, 0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0], dtype=float)

sweep_trials = 10
sweep_n_symbols = 100
sweep_sps = ExperimentConfig.samples_per_symbol
sweep_rolloff = ExperimentConfig.rolloff
sweep_span = ExperimentConfig.rrc_span_symbols
sweep_phase_shift_deg = ExperimentConfig.phase_shift_deg

print('phase_shift_deg =', sweep_phase_shift_deg)

## Channel Selector

Run this cell with `analysis_n_rx = 1`, `2`, or `4` to reproduce experiment-style plots for each checkpoint.

In [ ]:
analysis_n_rx = 1
model, checkpoint_path = load_separator(analysis_n_rx)
print('loaded checkpoint:', checkpoint_path)

## Unnormalized Noise Sweep

This reproduces the familiar experiment style: raw `sigma2` is used directly as an absolute complex sample variance.

In [ ]:
unnorm_soi_grid, unnorm_int_grid = run_symbol_error_sweep(
    model=model,
    n_rx=analysis_n_rx,
    phase_shift_deg=sweep_phase_shift_deg,
    alpha_values=alpha_values,
    sigma2_values=unnormalized_sigma2_values,
    trials=sweep_trials,
    n_symbols=sweep_n_symbols,
    sps=sweep_sps,
    rolloff=sweep_rolloff,
    span=sweep_span,
    noise_mode='unnormalized',
)

In [ ]:
sigma2_grid_u, alpha_grid_u = np.meshgrid(unnormalized_sigma2_values, alpha_values)
fig = plt.figure(figsize=(16, 7), constrained_layout=True)

ax1 = fig.add_subplot(121, projection='3d')
surf1 = ax1.plot_surface(sigma2_grid_u, alpha_grid_u, unnorm_soi_grid, cmap='viridis', edgecolor='none')
ax1.set_xlabel('sigma^2')
ax1.set_ylabel('alpha')
ax1.set_zlabel('SOI incorrect symbols')
ax1.set_zlim(0, 100)
ax1.set_title(f'{analysis_n_rx} Channel SOI Mean Incorrect Symbols Sweep')
ax1.view_init(elev=25, azim=225)
fig.colorbar(surf1, ax=ax1, shrink=0.7, pad=0.08)

ax2 = fig.add_subplot(122, projection='3d')
surf2 = ax2.plot_surface(sigma2_grid_u, alpha_grid_u, unnorm_int_grid, cmap='viridis', edgecolor='none')
ax2.set_xlabel('sigma^2')
ax2.set_ylabel('alpha')
ax2.set_zlabel('interferer incorrect symbols')
ax2.set_zlim(0, 100)
ax2.set_title(f'{analysis_n_rx} Channel Interferer Mean Incorrect Symbols Sweep')
ax2.view_init(elev=25, azim=225)
fig.colorbar(surf2, ax=ax2, shrink=0.7, pad=0.08)

plt.show()

## Normalized Noise Sweep

This keeps the same experiment-style symbol-error surfaces, but the y-axis now represents a **normalized** noise ratio relative to clean waveform power rather than an absolute sample variance.

In [ ]:
norm_soi_grid, norm_int_grid = run_symbol_error_sweep(
    model=model,
    n_rx=analysis_n_rx,
    phase_shift_deg=sweep_phase_shift_deg,
    alpha_values=alpha_values,
    sigma2_values=normalized_sigma2_values,
    trials=sweep_trials,
    n_symbols=sweep_n_symbols,
    sps=sweep_sps,
    rolloff=sweep_rolloff,
    span=sweep_span,
    noise_mode='normalized',
)

In [ ]:
sigma2_grid_n, alpha_grid_n = np.meshgrid(normalized_sigma2_values, alpha_values)
fig = plt.figure(figsize=(16, 7), constrained_layout=True)

ax1 = fig.add_subplot(121, projection='3d')
surf1 = ax1.plot_surface(sigma2_grid_n, alpha_grid_n, norm_soi_grid, cmap='viridis', edgecolor='none')
ax1.set_xlabel('normalized sigma^2 ratio')
ax1.set_ylabel('alpha')
ax1.set_zlabel('SOI incorrect symbols')
ax1.set_zlim(0, 100)
ax1.set_title(f'{analysis_n_rx} Channel SOI Mean Incorrect Symbols Sweep')
ax1.view_init(elev=25, azim=225)
fig.colorbar(surf1, ax=ax1, shrink=0.7, pad=0.08)

ax2 = fig.add_subplot(122, projection='3d')
surf2 = ax2.plot_surface(sigma2_grid_n, alpha_grid_n, norm_int_grid, cmap='viridis', edgecolor='none')
ax2.set_xlabel('normalized sigma^2 ratio')
ax2.set_ylabel('alpha')
ax2.set_zlabel('interferer incorrect symbols')
ax2.set_zlim(0, 100)
ax2.set_title(f'{analysis_n_rx} Channel Interferer Mean Incorrect Symbols Sweep')
ax2.view_init(elev=25, azim=225)
fig.colorbar(surf2, ax=ax2, shrink=0.7, pad=0.08)

plt.show()

## 2D Alpha Plot At Several Normalized Noise Levels

This mirrors the experiment outputs more directly by showing how symbol error changes with `alpha` under several normalized noise ratios.

In [ ]:
plt.figure(figsize=(10, 6), constrained_layout=True)
for normalized_sigma2 in [0.01, 0.1, 0.5, 1.0, 2.0]:
    idx = int(np.where(np.isclose(normalized_sigma2_values, normalized_sigma2))[0][0])
    plt.plot(alpha_values, norm_soi_grid[:, idx], linewidth=2, label=f'SOI norm sigma^2={normalized_sigma2:g}')

plt.xlabel('alpha')
plt.ylabel('mean incorrect symbols')
plt.title(f'{analysis_n_rx} Channel SOI Error Vs Alpha At Normalized Noise Levels')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()